# 06 Lagged R2G and Granger Directionality Checks

This notebook supplements the contemporaneous connectedness notebooks with temporal-direction checks. It estimates pairwise Granger predictive direction and lagged strict R2G contributions for the same seven networks used in notebooks 04 and 05.

In [1]:

from __future__ import annotations

import hashlib
import json
import platform
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import statsmodels.api as sm

warnings.filterwarnings("ignore", category=RuntimeWarning)

PROJECT_ROOT = Path.cwd()
INPUT_PATH = PROJECT_ROOT / "data" / "processed" / "02_iceemdan_hierarchical_ward" / "hierarchical_ward_components_wide.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "06_lagged_r2g_granger_directionality"
VIS_DIR = OUTPUT_DIR / "visualizations"
GRANGER_NETWORK_DIR = VIS_DIR / "granger_networks"
LAGGED_HEATMAP_DIR = VIS_DIR / "lagged_r2g_heatmaps"
LAGGED_NETWORK_DIR = VIS_DIR / "lagged_npdc_networks"
for path in [OUTPUT_DIR, VIS_DIR, GRANGER_NETWORK_DIR, LAGGED_HEATMAP_DIR, LAGGED_NETWORK_DIR]:
    path.mkdir(parents=True, exist_ok=True)

METAL_ORDER = ["gold", "silver", "platinum", "palladium"]
MARKET_ORDER = ["futures", "spot"]
SCALE_ORDER = ["STC", "MTC", "LTC"]
RETURN_SERIES_ORDER = [
    "gold_futures", "silver_futures", "platinum_futures", "palladium_futures",
    "gold_spot", "silver_spot", "platinum_spot", "palladium_spot",
]
EXPECTED_COLUMNS = ["date", *[f"{series}_{scale}" for series in RETURN_SERIES_ORDER for scale in SCALE_ORDER]]
PARAMS = {
    "method": "Pairwise Granger tests plus lagged strict R2G contribution decomposition",
    "max_lag": 5,
    "lag_selection": "BIC",
    "fdr_method": "Benjamini-Hochberg",
    "fdr_alpha": 0.05,
    "r2_tolerance": 1e-8,
    "eigenvalue_tolerance": 1e-12,
}

plt.rcParams.update({"figure.dpi": 140, "savefig.dpi": 300, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
start_time = time.perf_counter()


## Input and Network Definitions

In [2]:

def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


components_wide = pd.read_csv(INPUT_PATH, parse_dates=["date"])
assert components_wide.shape == (3100, 25), components_wide.shape
assert list(components_wide.columns) == EXPECTED_COLUMNS
assert components_wide["date"].min() == pd.Timestamp("2012-01-04")
assert components_wide["date"].max() == pd.Timestamp("2026-03-31")
assert components_wide.drop(columns=["date"]).isna().sum().sum() == 0
assert np.isfinite(components_wide.drop(columns=["date"]).to_numpy()).all()

input_checks = {
    "input_file": str(INPUT_PATH.relative_to(PROJECT_ROOT)),
    "input_sha256": sha256_file(INPUT_PATH),
    "n_rows": int(len(components_wide)),
    "n_component_columns": int(len(components_wide.columns) - 1),
    "date_start": components_wide["date"].min().strftime("%Y-%m-%d"),
    "date_end": components_wide["date"].max().strftime("%Y-%m-%d"),
}


@dataclass(frozen=True)
class NetworkDefinition:
    network_id: str
    network_type: str
    node_columns: tuple[str, ...]
    scale: str | None = None
    metal: str | None = None


def parse_component_column(column: str) -> dict[str, str]:
    series, scale = column.rsplit("_", 1)
    metal, market = series.rsplit("_", 1)
    return {"series": series, "metal": metal, "market": market, "scale": scale}


def display_label(column: str, network: NetworkDefinition) -> str:
    meta = parse_component_column(column)
    if network.network_type == "same_scale_cross_metal":
        return f"{meta['metal'].title()} {meta['market'].title()}"
    return f"{meta['market'].title()} {meta['scale']}"


networks: list[NetworkDefinition] = []
for scale in SCALE_ORDER:
    networks.append(
        NetworkDefinition(
            network_id=f"same_scale_{scale}",
            network_type="same_scale_cross_metal",
            scale=scale,
            node_columns=tuple(f"{series}_{scale}" for series in RETURN_SERIES_ORDER),
        )
    )
for metal in METAL_ORDER:
    node_columns = []
    for scale in SCALE_ORDER:
        for market in MARKET_ORDER:
            node_columns.append(f"{metal}_{market}_{scale}")
    networks.append(
        NetworkDefinition(
            network_id=f"within_{metal}",
            network_type="within_metal_cross_scale",
            metal=metal,
            node_columns=tuple(node_columns),
        )
    )

network_metadata_rows = []
for network in networks:
    for order, column in enumerate(network.node_columns, start=1):
        network_metadata_rows.append(
            {
                "network_id": network.network_id,
                "network_type": network.network_type,
                "network_scale": network.scale,
                "network_metal": network.metal,
                "node_order": order,
                "node_id": column,
                "display_label": display_label(column, network),
                "component_column": column,
                **parse_component_column(column),
            }
        )
network_node_metadata = pd.DataFrame(network_metadata_rows)
network_node_metadata.to_csv(OUTPUT_DIR / "network_node_metadata.csv", index=False)
assert len(networks) == 7
network_node_metadata.head()


,network_id,network_type,network_scale,network_metal,node_order,node_id,display_label,component_column,series,metal,market,scale
0,same_scale_STC,same_scale_cross_metal,STC,NaN,1,gold_futures_STC,Gold Futures,gold_futures_STC,gold_futures,gold,futures,STC
1,same_scale_STC,same_scale_cross_metal,STC,NaN,2,silver_futures_STC,Silver Futures,silver_futures_STC,silver_futures,silver,futures,STC
2,same_scale_STC,same_scale_cross_metal,STC,NaN,3,platinum_futures_STC,Platinum Futures,platinum_futures_STC,platinum_futures,platinum,futures,STC
3,same_scale_STC,same_scale_cross_metal,STC,NaN,4,palladium_futures_STC,Palladium Futures,palladium_futures_STC,palladium_futures,palladium,futures,STC
4,same_scale_STC,same_scale_cross_metal,STC,NaN,5,gold_spot_STC,Gold Spot,gold_spot_STC,gold_spot,gold,spot,STC


## Helpers

In [3]:

def fdr_bh(p_values: pd.Series, alpha: float = 0.05) -> tuple[np.ndarray, np.ndarray]:
    p = p_values.to_numpy(dtype=float)
    n = len(p)
    order = np.argsort(p)
    ranked = p[order]
    adjusted_sorted = ranked * n / np.arange(1, n + 1)
    adjusted_sorted = np.minimum.accumulate(adjusted_sorted[::-1])[::-1]
    adjusted_sorted = np.clip(adjusted_sorted, 0.0, 1.0)
    adjusted = np.empty(n, dtype=float)
    adjusted[order] = adjusted_sorted
    significant = adjusted <= alpha
    return adjusted, significant


def lagged_frame(df: pd.DataFrame, target: str, sources: list[str], lag: int) -> pd.DataFrame:
    out = pd.DataFrame({"y": df[target]})
    for ell in range(1, lag + 1):
        out[f"{target}_lag{ell}"] = df[target].shift(ell)
        for source in sources:
            out[f"{source}_lag{ell}"] = df[source].shift(ell)
    return out.dropna()


def ols_fit(y: pd.Series, x: pd.DataFrame):
    return sm.OLS(y.to_numpy(dtype=float), sm.add_constant(x.to_numpy(dtype=float), has_constant="add")).fit()


def granger_pair(df: pd.DataFrame, source: str, target: str, max_lag: int) -> dict[str, Any]:
    lag_rows = []
    fitted_by_lag = {}
    for lag in range(1, max_lag + 1):
        data = lagged_frame(df, target, [source], lag)
        y = data["y"]
        source_cols = [f"{source}_lag{ell}" for ell in range(1, lag + 1)]
        target_cols = [f"{target}_lag{ell}" for ell in range(1, lag + 1)]
        full = ols_fit(y, data[target_cols + source_cols])
        lag_rows.append({"lag": lag, "aic": full.aic, "bic": full.bic, "n_obs": int(full.nobs)})
        fitted_by_lag[lag] = (full, target_cols, source_cols)
    selection = min(lag_rows, key=lambda row: row["bic"])
    selected_lag = int(selection["lag"])
    full, target_cols, source_cols = fitted_by_lag[selected_lag]
    restriction = np.zeros((len(source_cols), len(full.params)))
    for row_idx, col_idx in enumerate(range(1 + len(target_cols), 1 + len(target_cols) + len(source_cols))):
        restriction[row_idx, col_idx] = 1.0
    f_test = full.f_test(restriction)
    return {
        "source": source,
        "target": target,
        "selected_lag_bic": selected_lag,
        "selected_lag_aic": int(min(lag_rows, key=lambda row: row["aic"])["lag"]),
        "f_stat": float(np.asarray(f_test.fvalue).ravel()[0]),
        "p_value": float(np.asarray(f_test.pvalue).ravel()[0]),
        "n_obs": int(full.nobs),
        "full_model_aic": float(full.aic),
        "full_model_bic": float(full.bic),
    }


def nearest_psd(matrix: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    matrix = (np.asarray(matrix, dtype=float) + np.asarray(matrix, dtype=float).T) / 2.0
    eigvals, eigvecs = np.linalg.eigh(matrix)
    eigvals = np.maximum(eigvals, eps)
    repaired = (eigvecs * eigvals) @ eigvecs.T
    repaired = (repaired + repaired.T) / 2.0
    diag = np.sqrt(np.maximum(np.diag(repaired), eps))
    repaired = repaired / np.outer(diag, diag)
    repaired = (repaired + repaired.T) / 2.0
    np.fill_diagonal(repaired, 1.0)
    return repaired


def strict_r2g_from_design(y: np.ndarray, x: np.ndarray, eigenvalue_tol: float = 1e-12) -> tuple[np.ndarray, float]:
    y = np.asarray(y, dtype=float)
    x = np.asarray(x, dtype=float)
    y_z = (y - y.mean()) / y.std(ddof=0)
    x_z = (x - x.mean(axis=0)) / x.std(axis=0, ddof=0)
    corr_xx = nearest_psd((x_z.T @ x_z) / len(x_z), eps=eigenvalue_tol)
    corr_yx = (x_z.T @ y_z) / len(y_z)
    eigvals, eigvecs = np.linalg.eigh(corr_xx)
    eigvals = np.maximum(eigvals, eigenvalue_tol)
    rxz = (eigvecs * np.sqrt(eigvals)) @ eigvecs.T
    beta = np.linalg.solve(rxz, corr_yx)
    contributions = (rxz**2) @ (beta**2)
    contributions = np.clip(contributions, 0.0, None)
    total_r2 = float(corr_yx @ np.linalg.solve(corr_xx, corr_yx))
    total_r2 = float(np.clip(total_r2, 0.0, 1.0))
    if contributions.sum() > eigenvalue_tol:
        contributions *= total_r2 / contributions.sum()
    return np.clip(contributions, 0.0, None), total_r2


def matrix_rows(matrix: np.ndarray, network: NetworkDefinition, selected_lags: dict[str, int]) -> list[dict[str, Any]]:
    rows = []
    for target_idx, target in enumerate(network.node_columns):
        for source_idx, source in enumerate(network.node_columns):
            if target_idx == source_idx:
                continue
            rows.append(
                {
                    "network_id": network.network_id,
                    "network_type": network.network_type,
                    "network_scale": network.scale,
                    "network_metal": network.metal,
                    "target": target,
                    "source": source,
                    "selected_lag": selected_lags[target],
                    "lagged_r2g": float(matrix[target_idx, source_idx]),
                    "lagged_r2g_percent": float(matrix[target_idx, source_idx] * 100.0),
                }
            )
    return rows


def directional_rows(matrix: np.ndarray, network: NetworkDefinition) -> list[dict[str, Any]]:
    from_values = matrix.sum(axis=1) * 100.0
    to_values = matrix.sum(axis=0) * 100.0
    return [
        {
            "network_id": network.network_id,
            "network_type": network.network_type,
            "network_scale": network.scale,
            "network_metal": network.metal,
            "node": node,
            "FROM": float(from_values[idx]),
            "TO": float(to_values[idx]),
            "NET": float(to_values[idx] - from_values[idx]),
        }
        for idx, node in enumerate(network.node_columns)
    ]


def npdc_rows(matrix: np.ndarray, network: NetworkDefinition) -> list[dict[str, Any]]:
    rows = []
    for source_idx, source in enumerate(network.node_columns):
        for target_idx, target in enumerate(network.node_columns):
            if source_idx == target_idx:
                continue
            source_to_target = matrix[target_idx, source_idx] * 100.0
            target_to_source = matrix[source_idx, target_idx] * 100.0
            rows.append(
                {
                    "network_id": network.network_id,
                    "network_type": network.network_type,
                    "network_scale": network.scale,
                    "network_metal": network.metal,
                    "source": source,
                    "target": target,
                    "source_to_target": float(source_to_target),
                    "target_to_source": float(target_to_source),
                    "NPDC": float(source_to_target - target_to_source),
                }
            )
    return rows


## Pairwise Granger Tests

In [4]:

granger_rows = []
for network in networks:
    network_df = components_wide.loc[:, list(network.node_columns)]
    for source_node in network.node_columns:
        for target_node in network.node_columns:
            if source_node == target_node:
                continue
            row = granger_pair(network_df, source_node, target_node, PARAMS["max_lag"])
            row.update(
                {
                    "network_id": network.network_id,
                    "network_type": network.network_type,
                    "network_scale": network.scale,
                    "network_metal": network.metal,
                }
            )
            granger_rows.append(row)

granger_pairwise_results = pd.DataFrame(granger_rows)
granger_pairwise_results["fdr_p_value"], granger_pairwise_results["significant_5pct"] = fdr_bh(
    granger_pairwise_results["p_value"],
    alpha=PARAMS["fdr_alpha"],
)
granger_pairwise_results = granger_pairwise_results[
    [
        "network_id", "network_type", "network_scale", "network_metal", "source", "target",
        "selected_lag_bic", "selected_lag_aic", "f_stat", "p_value", "fdr_p_value",
        "significant_5pct", "n_obs", "full_model_aic", "full_model_bic",
    ]
]
granger_pairwise_results.to_csv(OUTPUT_DIR / "granger_pairwise_results.csv", index=False)
granger_pairwise_results.loc[granger_pairwise_results["significant_5pct"]].to_csv(OUTPUT_DIR / "granger_significant_paths.csv", index=False)
granger_pairwise_results.groupby("network_id")["significant_5pct"].sum()


E:\@PythonProject\@Precious\precious-metal\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 5, but rank is 3
  warnings.warn('covariance of constraints does not have full '
E:\@PythonProject\@Precious\precious-metal\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 5, but rank is 3
  warnings.warn('covariance of constraints does not have full '
E:\@PythonProject\@Precious\precious-metal\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 4, but rank is 3
  warnings.warn('covariance of constraints does not have full '
E:\@PythonProject\@Precious\precious-metal\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of 

E:\@PythonProject\@Precious\precious-metal\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 5, but rank is 4
  warnings.warn('covariance of constraints does not have full '
E:\@PythonProject\@Precious\precious-metal\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 5, but rank is 4
  warnings.warn('covariance of constraints does not have full '
E:\@PythonProject\@Precious\precious-metal\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 5, but rank is 4
  warnings.warn('covariance of constraints does not have full '
E:\@PythonProject\@Precious\precious-metal\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of 

E:\@PythonProject\@Precious\precious-metal\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 5, but rank is 4
  warnings.warn('covariance of constraints does not have full '
E:\@PythonProject\@Precious\precious-metal\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 5, but rank is 4
  warnings.warn('covariance of constraints does not have full '
E:\@PythonProject\@Precious\precious-metal\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 5, but rank is 4
  warnings.warn('covariance of constraints does not have full '
E:\@PythonProject\@Precious\precious-metal\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of 

E:\@PythonProject\@Precious\precious-metal\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 5, but rank is 4
  warnings.warn('covariance of constraints does not have full '
E:\@PythonProject\@Precious\precious-metal\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 5, but rank is 4
  warnings.warn('covariance of constraints does not have full '
E:\@PythonProject\@Precious\precious-metal\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 5, but rank is 4
  warnings.warn('covariance of constraints does not have full '
E:\@PythonProject\@Precious\precious-metal\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of 

E:\@PythonProject\@Precious\precious-metal\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 5, but rank is 3
  warnings.warn('covariance of constraints does not have full '
E:\@PythonProject\@Precious\precious-metal\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 5, but rank is 3
  warnings.warn('covariance of constraints does not have full '
E:\@PythonProject\@Precious\precious-metal\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 5, but rank is 3
  warnings.warn('covariance of constraints does not have full '
E:\@PythonProject\@Precious\precious-metal\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of 

E:\@PythonProject\@Precious\precious-metal\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 5, but rank is 4
  warnings.warn('covariance of constraints does not have full '
E:\@PythonProject\@Precious\precious-metal\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 5, but rank is 4
  warnings.warn('covariance of constraints does not have full '
E:\@PythonProject\@Precious\precious-metal\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 5, but rank is 4
  warnings.warn('covariance of constraints does not have full '
E:\@PythonProject\@Precious\precious-metal\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of 

network_id
same_scale_LTC      29
same_scale_MTC      47
same_scale_STC      45
within_gold         17
within_palladium    15
within_platinum     18
within_silver       16
Name: significant_5pct, dtype: int64

## Lagged Strict R2G

In [5]:

lagged_matrix_rows = []
lagged_by_lag_rows = []
lagged_directional_rows = []
lagged_npdc_rows = []
lagged_tci_rows = []
lag_selection_rows = []
validation_rows = []
lagged_matrices: dict[str, np.ndarray] = {}

for network in networks:
    network_df = components_wide.loc[:, list(network.node_columns)]
    k = len(network.node_columns)
    matrix = np.zeros((k, k), dtype=float)
    selected_lags_by_target: dict[str, int] = {}

    for target_idx, target in enumerate(network.node_columns):
        sources = [node for node in network.node_columns if node != target]
        candidate_rows = []
        candidate_data: dict[int, tuple[pd.DataFrame, list[tuple[str, int, str]]]] = {}
        for lag in range(1, PARAMS["max_lag"] + 1):
            data = pd.DataFrame({"y": network_df[target]})
            predictors: list[tuple[str, int, str]] = []
            for source in sources:
                for ell in range(1, lag + 1):
                    col = f"{source}_lag{ell}"
                    data[col] = network_df[source].shift(ell)
                    predictors.append((source, ell, col))
            data = data.dropna()
            fit = ols_fit(data["y"], data[[col for _, _, col in predictors]])
            candidate_rows.append({"lag": lag, "aic": fit.aic, "bic": fit.bic, "n_obs": int(fit.nobs), "target": target, "network_id": network.network_id})
            candidate_data[lag] = (data, predictors)

        selected = min(candidate_rows, key=lambda row: row["bic"])
        selected_lag = int(selected["lag"])
        selected_lags_by_target[target] = selected_lag
        lag_selection_rows.extend(candidate_rows)
        data, predictors = candidate_data[selected_lag]
        x = data[[col for _, _, col in predictors]].to_numpy(dtype=float)
        y = data["y"].to_numpy(dtype=float)
        contributions, total_r2 = strict_r2g_from_design(y, x, PARAMS["eigenvalue_tolerance"])

        by_source = {source: 0.0 for source in sources}
        for contribution, (source, ell, col) in zip(contributions, predictors):
            by_source[source] += float(contribution)
            lagged_by_lag_rows.append(
                {
                    "network_id": network.network_id,
                    "network_type": network.network_type,
                    "network_scale": network.scale,
                    "network_metal": network.metal,
                    "target": target,
                    "source": source,
                    "lag": ell,
                    "selected_lag": selected_lag,
                    "lagged_r2g": float(contribution),
                    "lagged_r2g_percent": float(contribution * 100.0),
                }
            )
        for source, value in by_source.items():
            source_idx = list(network.node_columns).index(source)
            matrix[target_idx, source_idx] = value
        validation_rows.append(
            {
                "network_id": network.network_id,
                "target": target,
                "selected_lag": selected_lag,
                "total_r2": total_r2,
                "contribution_sum": float(contributions.sum()),
                "row_error": float(abs(contributions.sum() - total_r2)),
                "min_contribution": float(contributions.min()),
                "max_contribution": float(contributions.max()),
                "n_predictors": len(predictors),
                "n_obs": int(len(data)),
            }
        )

    lagged_matrices[network.network_id] = matrix
    lagged_matrix_rows.extend(matrix_rows(matrix, network, selected_lags_by_target))
    lagged_directional_rows.extend(directional_rows(matrix, network))
    lagged_npdc_rows.extend(npdc_rows(matrix, network))
    row_sums = matrix.sum(axis=1)
    lagged_tci_rows.append(
        {
            "network_id": network.network_id,
            "network_type": network.network_type,
            "network_scale": network.scale,
            "network_metal": network.metal,
            "TCI": float(row_sums.mean() * 100.0),
            "mean_total_r2": float(row_sums.mean()),
            "min_total_r2": float(row_sums.min()),
            "max_total_r2": float(row_sums.max()),
        }
    )

lag_selection_by_target = pd.DataFrame(lag_selection_rows)
lagged_r2g_matrices_long = pd.DataFrame(lagged_matrix_rows)
lagged_r2g_by_lag_long = pd.DataFrame(lagged_by_lag_rows)
lagged_directional_indices = pd.DataFrame(lagged_directional_rows)
lagged_npdc = pd.DataFrame(lagged_npdc_rows)
lagged_tci = pd.DataFrame(lagged_tci_rows)
lagged_validation = pd.DataFrame(validation_rows)

assert lagged_validation["row_error"].max() < PARAMS["r2_tolerance"]
assert lagged_r2g_matrices_long["lagged_r2g"].min() >= -PARAMS["r2_tolerance"]
assert lagged_tci["TCI"].between(0, 100).all()

lag_selection_by_target.to_csv(OUTPUT_DIR / "lag_selection_by_target.csv", index=False)
lagged_r2g_matrices_long.to_csv(OUTPUT_DIR / "lagged_r2g_matrices_long.csv", index=False)
lagged_r2g_by_lag_long.to_csv(OUTPUT_DIR / "lagged_r2g_by_lag_long.csv", index=False)
lagged_directional_indices.to_csv(OUTPUT_DIR / "lagged_directional_indices.csv", index=False)
lagged_npdc.to_csv(OUTPUT_DIR / "lagged_npdc.csv", index=False)
lagged_tci.to_csv(OUTPUT_DIR / "lagged_tci.csv", index=False)
lagged_validation.to_csv(OUTPUT_DIR / "lagged_r2g_validation.csv", index=False)
lagged_tci


,network_id,network_type,network_scale,network_metal,TCI,mean_total_r2,min_total_r2,max_total_r2
0,same_scale_STC,same_scale_cross_metal,STC,NaN,14.963373,0.149634,0.090786,0.227363
1,same_scale_MTC,same_scale_cross_metal,MTC,NaN,70.809551,0.708096,0.268870,0.889441
2,same_scale_LTC,same_scale_cross_metal,LTC,NaN,85.909300,0.859093,0.751064,0.943825
3,within_gold,within_metal_cross_scale,NaN,gold,54.075640,0.540756,0.141488,0.767467
4,within_silver,within_metal_cross_scale,NaN,silver,49.566552,0.495666,0.084558,0.816686
5,within_platinum,within_metal_cross_scale,NaN,platinum,62.776256,0.627763,0.088826,0.929887
6,within_palladium,within_metal_cross_scale,NaN,palladium,58.908260,0.589083,0.071782,0.907358


## Visualizations

In [6]:

def save_figure(fig: plt.Figure, directory: Path, filename_stem: str) -> None:
    fig.tight_layout()
    fig.savefig(directory / f"{filename_stem}.png", bbox_inches="tight")
    fig.savefig(directory / f"{filename_stem}.pdf", bbox_inches="tight")
    plt.close(fig)


def plot_lagged_heatmap(network: NetworkDefinition, matrix: np.ndarray) -> None:
    labels = [display_label(column, network) for column in network.node_columns]
    fig, ax = plt.subplots(figsize=(8.2, 6.4))
    im = ax.imshow(matrix * 100.0, cmap="YlGnBu", aspect="auto")
    ax.set_title(f"Lagged strict R2G matrix: {network.network_id}")
    ax.set_xlabel("Lagged source")
    ax.set_ylabel("Target")
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels)
    max_value = np.nanmax(matrix * 100.0)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = matrix[i, j] * 100.0
            text_color = "white" if value > max_value * 0.55 else "black"
            ax.text(j, i, f"{value:.1f}", ha="center", va="center", fontsize=7, color=text_color)
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Lagged R2G contribution (%)")
    save_figure(fig, LAGGED_HEATMAP_DIR, f"{network.network_id}_lagged_r2g_heatmap")


def plot_npdc_network(network: NetworkDefinition, npdc_df: pd.DataFrame, directory: Path, filename_stem: str, title: str) -> None:
    data = npdc_df.loc[npdc_df["network_id"].eq(network.network_id) & npdc_df["NPDC"].gt(0)].copy()
    data = data.sort_values("NPDC", ascending=False).head(min(14, len(data)))
    label_map = {column: display_label(column, network) for column in network.node_columns}
    graph = nx.DiGraph()
    for node in network.node_columns:
        graph.add_node(node, label=label_map[node])
    for _, row in data.iterrows():
        graph.add_edge(row["source"], row["target"], weight=row["NPDC"])
    fig, ax = plt.subplots(figsize=(8.2, 6.2))
    pos = nx.circular_layout(graph)
    weights = [graph[u][v]["weight"] for u, v in graph.edges()]
    widths = [0.8 + 3.2 * weight / max(weights) for weight in weights] if weights else []
    nx.draw_networkx_nodes(graph, pos, node_size=1550, node_color="#d8edf3", edgecolors="#305f72", linewidths=1.0, ax=ax)
    nx.draw_networkx_labels(graph, pos, labels=label_map, font_size=7, ax=ax)
    nx.draw_networkx_edges(graph, pos, width=widths, edge_color="#1f77b4", arrows=True, arrowsize=15, connectionstyle="arc3,rad=0.08", ax=ax)
    edge_labels = {(u, v): f"{graph[u][v]['weight']:.1f}" for u, v in graph.edges()}
    nx.draw_networkx_edge_labels(graph, pos, edge_labels=edge_labels, font_size=7, ax=ax)
    ax.set_title(title)
    ax.axis("off")
    save_figure(fig, directory, filename_stem)


def plot_granger_network(network: NetworkDefinition) -> None:
    data = granger_pairwise_results.loc[
        granger_pairwise_results["network_id"].eq(network.network_id)
        & granger_pairwise_results["significant_5pct"]
    ].copy()
    if data.empty:
        return
    data = data.sort_values(["fdr_p_value", "p_value"]).head(min(14, len(data)))
    label_map = {column: display_label(column, network) for column in network.node_columns}
    graph = nx.DiGraph()
    for node in network.node_columns:
        graph.add_node(node, label=label_map[node])
    for _, row in data.iterrows():
        graph.add_edge(row["source"], row["target"], weight=-np.log10(max(row["fdr_p_value"], 1e-300)))
    fig, ax = plt.subplots(figsize=(8.2, 6.2))
    pos = nx.circular_layout(graph)
    weights = [graph[u][v]["weight"] for u, v in graph.edges()]
    widths = [0.8 + 3.0 * weight / max(weights) for weight in weights] if weights else []
    nx.draw_networkx_nodes(graph, pos, node_size=1550, node_color="#f7e2bf", edgecolors="#8a5a14", linewidths=1.0, ax=ax)
    nx.draw_networkx_labels(graph, pos, labels=label_map, font_size=7, ax=ax)
    nx.draw_networkx_edges(graph, pos, width=widths, edge_color="#9c6b18", arrows=True, arrowsize=15, connectionstyle="arc3,rad=0.08", ax=ax)
    ax.set_title(f"FDR-significant Granger paths: {network.network_id}")
    ax.axis("off")
    save_figure(fig, GRANGER_NETWORK_DIR, f"{network.network_id}_granger_network")


for network in networks:
    plot_lagged_heatmap(network, lagged_matrices[network.network_id])
    plot_npdc_network(network, lagged_npdc, LAGGED_NETWORK_DIR, f"{network.network_id}_lagged_npdc_network", f"Lagged R2G NPDC network: {network.network_id}")
    plot_granger_network(network)

print("Lagged R2G and Granger visualizations saved.")


Lagged R2G and Granger visualizations saved.


## Manifest and Final Checks

In [7]:

def relpath(path: Path) -> str:
    return str(path.relative_to(PROJECT_ROOT)).replace("\\", "/")


required_outputs = [
    OUTPUT_DIR / "network_node_metadata.csv",
    OUTPUT_DIR / "granger_pairwise_results.csv",
    OUTPUT_DIR / "granger_significant_paths.csv",
    OUTPUT_DIR / "lag_selection_by_target.csv",
    OUTPUT_DIR / "lagged_r2g_matrices_long.csv",
    OUTPUT_DIR / "lagged_r2g_by_lag_long.csv",
    OUTPUT_DIR / "lagged_directional_indices.csv",
    OUTPUT_DIR / "lagged_npdc.csv",
    OUTPUT_DIR / "lagged_tci.csv",
    OUTPUT_DIR / "lagged_r2g_validation.csv",
]
for output_file in required_outputs:
    assert output_file.exists(), output_file

figure_counts = {
    "granger_network_png": len(list(GRANGER_NETWORK_DIR.glob("*.png"))),
    "lagged_heatmap_png": len(list(LAGGED_HEATMAP_DIR.glob("*.png"))),
    "lagged_network_png": len(list(LAGGED_NETWORK_DIR.glob("*.png"))),
}
assert figure_counts["lagged_heatmap_png"] == len(networks)
assert figure_counts["lagged_network_png"] == len(networks)

manifest = {
    "method": "Lagged strict R2G and Granger directionality checks on Hierarchical Ward STC/MTC/LTC reconstructed components.",
    "method_note": "Pairwise Granger tests assess lagged predictive direction. Lagged R2G decomposes target components by lagged source components using the strict R2G formula; it should be interpreted as lagged explanatory direction, not structural shock transmission.",
    "parameters": PARAMS,
    "input_checks": input_checks,
    "networks": [
        {
            "network_id": network.network_id,
            "network_type": network.network_type,
            "scale": network.scale,
            "metal": network.metal,
            "node_columns": list(network.node_columns),
        }
        for network in networks
    ],
    "validation": {
        "max_lagged_r2g_row_error": float(lagged_validation["row_error"].max()),
        "n_granger_tests": int(len(granger_pairwise_results)),
        "n_granger_significant_fdr_5pct": int(granger_pairwise_results["significant_5pct"].sum()),
        "figure_counts": figure_counts,
    },
    "outputs": {
        "network_node_metadata": relpath(OUTPUT_DIR / "network_node_metadata.csv"),
        "granger_pairwise_results": relpath(OUTPUT_DIR / "granger_pairwise_results.csv"),
        "granger_significant_paths": relpath(OUTPUT_DIR / "granger_significant_paths.csv"),
        "lag_selection_by_target": relpath(OUTPUT_DIR / "lag_selection_by_target.csv"),
        "lagged_r2g_matrices_long": relpath(OUTPUT_DIR / "lagged_r2g_matrices_long.csv"),
        "lagged_r2g_by_lag_long": relpath(OUTPUT_DIR / "lagged_r2g_by_lag_long.csv"),
        "lagged_directional_indices": relpath(OUTPUT_DIR / "lagged_directional_indices.csv"),
        "lagged_npdc": relpath(OUTPUT_DIR / "lagged_npdc.csv"),
        "lagged_tci": relpath(OUTPUT_DIR / "lagged_tci.csv"),
        "lagged_r2g_validation": relpath(OUTPUT_DIR / "lagged_r2g_validation.csv"),
        "visualizations": relpath(VIS_DIR),
    },
    "environment": {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "statsmodels": sm.__version__,
        "networkx": nx.__version__,
    },
    "runtime_seconds": float(time.perf_counter() - start_time),
}
manifest_path = OUTPUT_DIR / "lagged_r2g_granger_directionality_manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

print(json.dumps(manifest["validation"], ensure_ascii=False, indent=2))
print(f"Manifest saved to {relpath(manifest_path)}")


{
  "max_lagged_r2g_row_error": 3.3306690738754696e-16,
  "n_granger_tests": 288,
  "n_granger_significant_fdr_5pct": 187,
  "figure_counts": {
    "granger_network_png": 7,
    "lagged_heatmap_png": 7,
    "lagged_network_png": 7
  }
}
Manifest saved to data/processed/06_lagged_r2g_granger_directionality/lagged_r2g_granger_directionality_manifest.json
